# Filtered temperature Arrhenius fits

In [4]:
import os
import json
import numpy as np
import matplotlib

# ==========================================================
# NON-INTERACTIVE BACKEND
# ==========================================================

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy import stats
from collections import defaultdict


# ==========================================================
# CONSTANTS
# ==========================================================

kB = 8.617333262e-5       # eV/K
h = 4.135667696e-15       # eV*s


# ==========================================================
# INPUT - USE EXTRACTED DATA
# ==========================================================

JSON_FILE = "extracted_essential_data.json"
PLOTS_DIR = "Arrhenius_Eyring_Plots"

if not os.path.exists(PLOTS_DIR):

    os.makedirs(PLOTS_DIR)

    print(
        f"Created directory: {PLOTS_DIR}"
    )

else:

    print(
        f"Using existing directory: {PLOTS_DIR}"
    )


# ==========================================================
# POTENTIAL COLORS
# ==========================================================

POTENTIAL_COLORS = {

    "DND-BN": "blue",

    "JW": "orange",

    "Marinica": "green",

    "Unknown": "black"

}


# ==========================================================
# IDENTIFY POTENTIAL
# ==========================================================

def identify_potential(cascadeName):

    name = cascadeName.lower()

    if (
        "dnd-bn" in name
        or
        "dndbn" in name
    ):

        return "DND-BN"

    elif "jw" in name:

        return "JW"

    elif (
        "marinica" in name
        or
        "m-s" in name
        or
        "ms_" in name
        or
        name.startswith("ms")
    ):

        return "Marinica"

    else:

        return "Unknown"


# ==========================================================
# MATPLOTLIB SETTINGS
# ==========================================================

plt.rcParams.update({

    "font.family": "sans-serif",

    "font.sans-serif": [
        "Arial",
        "Helvetica",
        "DejaVu Sans"
    ],

    "font.size": 10,

    "axes.labelsize": 12,

    "axes.titlesize": 12,

    "xtick.labelsize": 10,

    "ytick.labelsize": 10,

    "figure.dpi": 300,

    "savefig.dpi": 600,

    "savefig.bbox": "tight",

    "savefig.pad_inches": 0.05,

    "axes.linewidth": 0.8,

    "axes.facecolor": "white",

    "figure.facecolor": "white",

    "lines.linewidth": 1.2,

    "ps.fonttype": 42,

    "pdf.fonttype": 42,

    "text.usetex": False

})


# ==========================================================
# FIGURE SIZE
# ==========================================================

FIG_WIDTH = 6.8
FIG_HEIGHT = 5.0


# ==========================================================
# FUNCTION: FORCE OPAQUE
# ==========================================================

def force_opaque(fig):

    fig.patch.set_facecolor("white")
    fig.patch.set_alpha(1.0)

    for ax in fig.axes:

        ax.set_facecolor("white")
        ax.patch.set_alpha(1.0)

        # --------------------------------------------------
        # Lines
        # --------------------------------------------------

        for line in ax.lines:

            line.set_alpha(1.0)

        # --------------------------------------------------
        # Scatter collections
        # --------------------------------------------------

        for collection in ax.collections:

            try:

                collection.set_alpha(1.0)

            except Exception:

                pass

        # --------------------------------------------------
        # Spines
        # --------------------------------------------------

        for spine in ax.spines.values():

            try:

                spine.set_alpha(1.0)

            except Exception:

                pass

        # --------------------------------------------------
        # Text
        # --------------------------------------------------

        for text in ax.texts:

            try:

                text.set_alpha(1.0)

            except Exception:

                pass

        # --------------------------------------------------
        # Grid lines
        # --------------------------------------------------

        for gridline in (
            ax.get_xgridlines()
            +
            ax.get_ygridlines()
        ):

            try:

                gridline.set_alpha(1.0)

            except Exception:

                pass


# ==========================================================
# FUNCTION: SAVE FIGURE
# ==========================================================

def save_publication_figure(
    fig,
    base_path,
    dpi=600
):

    force_opaque(fig)

    for fmt in [
        "png",
        "eps",
        "pdf"
    ]:

        path = f"{base_path}.{fmt}"

        fig.savefig(
            path,
            format=fmt,
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
            edgecolor="white",
            transparent=False
        )

        print(
            f"   Saved {fmt.upper()}: {path}"
        )


# ==========================================================
# FUNCTION: VALIDITY ASSESSMENT
# ==========================================================

def assess_validity(
    Ea_rel_error,
    DeltaH_rel_error,
    DeltaS_rel_error,
    A
):

    issues = []
    valid = True

    # ======================================================
    # Ea
    # ======================================================

    if Ea_rel_error < 10:

        Ea_status = "Excellent"

    elif Ea_rel_error < 20:

        Ea_status = "Good"

    elif Ea_rel_error < 35:

        Ea_status = "Acceptable"

    else:

        Ea_status = "Poor"

        issues.append(
            f"Ea error is "
            f"{Ea_rel_error:.1f}% (>35%)"
        )

        valid = False

    # ======================================================
    # Delta H
    # ======================================================

    if DeltaH_rel_error < 10:

        DeltaH_status = "Excellent"

    elif DeltaH_rel_error < 25:

        DeltaH_status = "Good"

    elif DeltaH_rel_error < 50:

        DeltaH_status = "Acceptable"

    else:

        DeltaH_status = "Poor"

        issues.append(
            f"DeltaH error is "
            f"{DeltaH_rel_error:.1f}% (>50%)"
        )

        valid = False

    # ======================================================
    # ARRHENIUS PREFACTOR
    # ======================================================

    log10_A = np.log10(A)

    if (
        12.0
        <=
        log10_A
        <
        14.0
    ):

        A_status = (
            f"log10(A) in 12-14 range "
            f"({log10_A:.3f})"
        )

        DeltaS_status = (
            "SKIPPED "
            "(log10(A) in 12-14 range)"
        )

    else:

        A_status = (
            f"log10(A) outside 12-14 range "
            f"({log10_A:.3f})"
        )

        if DeltaS_rel_error < 15:

            DeltaS_status = "Excellent"

        elif DeltaS_rel_error < 30:

            DeltaS_status = "Good"

        elif DeltaS_rel_error < 50:

            DeltaS_status = "Acceptable"

        else:

            DeltaS_status = "Poor"

            issues.append(
                f"DeltaS error is "
                f"{DeltaS_rel_error:.1f}% (>50%)"
            )

            valid = False

    # ======================================================
    # OVERALL
    # ======================================================

    if len(issues) == 0:

        overall = "VALID"

    else:

        overall = "INVALID"
        valid = False

    return {

        "overall":
            overall,

        "valid":
            valid,

        "Ea_status":
            Ea_status,

        "DeltaH_status":
            DeltaH_status,

        "DeltaS_status":
            DeltaS_status,

        "A_status":
            A_status,

        "issues":
            issues

    }


# ==========================================================
# MAIN PROCESSING
# ==========================================================

print()

print("=" * 75)

print(
    "STEP 2: ARRHENIUS/EYRING ANALYSIS ON EXTRACTED DATA"
)

print("=" * 75)


# ==========================================================
# CHECK INPUT FILE
# ==========================================================

if not os.path.exists(JSON_FILE):

    print(
        f"ERROR: {JSON_FILE} not found!"
    )

    print(
        "Please run the extraction script first."
    )

    exit(1)


# ==========================================================
# FILE SIZE
# ==========================================================

file_size = (
    os.path.getsize(JSON_FILE)
    /
    (1024 * 1024)
)

print(
    f"Input file size: "
    f"{file_size:.2f} MB"
)


# ==========================================================
# READ JSON
# ==========================================================

print(
    f"Reading {JSON_FILE}..."
)

with open(
    JSON_FILE,
    "r"
) as f:

    allData = json.load(f)

print(
    f"Total cascades found: "
    f"{len(allData)}"
)

print()


# ==========================================================
# STORAGE
# ==========================================================

all_results = {}

processed_count = 0
skipped_count = 0
error_count = 0


# ==========================================================
# PROCESS EACH CASCADE
# ==========================================================

for cascadeName, cascade in allData.items():

    processed_count += 1

    print()

    print("=" * 75)

    print(
        f"Processing "
        f"({processed_count}/{len(allData)}): "
        f"{cascadeName}"
    )

    print("=" * 75)

    try:

        # ==================================================
        # IDENTIFY POTENTIAL
        # ==================================================

        potential = identify_potential(
            cascadeName
        )

        potential_color = POTENTIAL_COLORS.get(
            potential,
            "black"
        )

        print(
            f"Potential: {potential} "
            f"(color: {potential_color})"
        )

        # ==================================================
        # GET DATA
        # ==================================================

        temps = cascade.get(
            "temp",
            []
        )

        flipTimes = cascade.get(
            "flipTime",
            []
        )

        fnames = cascade.get(
            "fname",
            []
        )

        rnds = cascade.get(
            "rnd",
            []
        )

        if len(temps) == 0:

            print(
                "No data found, skipping..."
            )

            skipped_count += 1

            continue

        print(
            f"Total trajectories: "
            f"{len(temps)}"
        )

        # ==================================================
        # GROUP BY TEMPERATURE
        # ==================================================

        temp_data = defaultdict(list)

        for T, flipTime, rnd in zip(
            temps,
            flipTimes,
            rnds
        ):

            try:

                T_int = int(T)

                temp_data[T_int].append(
                    (
                        flipTime,
                        rnd
                    )
                )

            except (
                ValueError,
                TypeError
            ):

                continue

        if len(temp_data) == 0:

            print(
                "No valid temperatures, skipping..."
            )

            skipped_count += 1

            continue

        # ==================================================
        # FILTER TEMPERATURES
        # ==================================================

        used_T = []
        skipped_T = []

        print()

        print(
            "Temperature statistics:"
        )

        print(
            "-" * 75
        )

        for T in sorted(
            temp_data.keys()
        ):

            samples = temp_data[T]

            total = len(
                samples
            )

            valid_count = sum(
                1
                for flip, rnd in samples
                if (
                    flip > 0
                    and
                    np.isfinite(flip)
                )
            )

            bad = (
                total
                -
                valid_count
            )

            status = (
                "USE"
                if
                bad <= 5
                else
                "SKIP"
            )

            if status == "USE":

                used_T.append(T)

            else:

                skipped_T.append(T)

            print(
                f"{T:5d} K : "
                f"Total={total:4d} "
                f"Valid={valid_count:4d} "
                f"Bad={bad:3d} "
                f"{status}"
            )

        # ==================================================
        # MINIMUM TEMPERATURES
        # ==================================================

        if len(used_T) < 3:

            print(
                f"Need at least 3 temperatures, "
                f"only {len(used_T)} available. "
                f"Skipping..."
            )

            skipped_count += 1

            continue

        # ==================================================
        # BUILD ARRAYS
        # ==================================================

        x_eyring = []
        y_eyring = []

        x_arrhenius = []
        y_arrhenius = []

        for T in used_T:

            for flipTime, rnd in temp_data[T]:

                if (
                    flipTime > 0
                    and
                    np.isfinite(flipTime)
                ):

                    omega = (
                        1.0
                        /
                        flipTime
                    )

                    # --------------------------------------
                    # EYRING
                    # --------------------------------------

                    x_eyring.append(
                        1.0 / T
                    )

                    y_eyring.append(
                        np.log(
                            omega / T
                        )
                    )

                    # --------------------------------------
                    # ARRHENIUS
                    # --------------------------------------

                    x_arrhenius.append(
                        1.0 / T
                    )

                    y_arrhenius.append(
                        np.log(omega)
                    )

        # ==================================================
        # CHECK DATA
        # ==================================================

        if len(x_eyring) < 3:

            print(
                "Not enough data points "
                "for fitting. Skipping..."
            )

            skipped_count += 1

            continue

        print(
            f"Data points for fitting: "
            f"{len(x_eyring)}"
        )

        # ==================================================
        # NUMPY ARRAYS
        # ==================================================

        x_eyring = np.asarray(
            x_eyring,
            dtype=float
        )

        y_eyring = np.asarray(
            y_eyring,
            dtype=float
        )

        x_arrhenius = np.asarray(
            x_arrhenius,
            dtype=float
        )

        y_arrhenius = np.asarray(
            y_arrhenius,
            dtype=float
        )

        # ==================================================
        # EYRING FIT
        # ==================================================

        (
            slope_eyring,
            intercept_eyring,
            r_eyring,
            p_eyring,
            std_eyring
        ) = stats.linregress(
            x_eyring,
            y_eyring
        )

        # ==================================================
        # ARRHENIUS FIT
        # ==================================================

        (
            slope_arrhenius,
            intercept_arrhenius,
            r_arrhenius,
            p_arrhenius,
            std_arrhenius
        ) = stats.linregress(
            x_arrhenius,
            y_arrhenius
        )

        # ==================================================
        # CALCULATE PARAMETERS
        # ==================================================

        DeltaH = (
            -kB
            *
            slope_eyring
        )

        DeltaS = (
            kB
            *
            (
                intercept_eyring
                -
                np.log(kB / h)
            )
        )

        Ea = (
            -slope_arrhenius
            *
            kB
        )

        A = np.exp(
            intercept_arrhenius
        )

        # ==================================================
        # EYRING ERROR ANALYSIS
        # ==================================================

        n = len(
            x_eyring
        )

        yfit = (
            slope_eyring
            *
            x_eyring
            +
            intercept_eyring
        )

        residuals = (
            y_eyring
            -
            yfit
        )

        sigma2 = (
            np.sum(
                residuals ** 2
            )
            /
            (n - 2)
            if n > 2
            else 0
        )

        sigma = (
            np.sqrt(
                sigma2
            )
            if sigma2 > 0
            else 0
        )

        x_mean = np.mean(
            x_eyring
        )

        sxx = np.sum(
            (
                x_eyring
                -
                x_mean
            ) ** 2
        )

        slope_error = (
            sigma
            /
            np.sqrt(sxx)
            if sxx > 0
            else 0
        )

        intercept_error = (
            sigma
            *
            np.sqrt(
                1.0 / n
                +
                x_mean**2
                /
                sxx
            )
            if sxx > 0
            else 0
        )

        DeltaH_error = (
            kB
            *
            slope_error
        )

        DeltaS_error = (
            kB
            *
            intercept_error
        )

        # ==================================================
        # ARRHENIUS ERROR ANALYSIS
        # ==================================================

        n_arr = len(
            x_arrhenius
        )

        yfit_arr = (
            slope_arrhenius
            *
            x_arrhenius
            +
            intercept_arrhenius
        )

        residuals_arr = (
            y_arrhenius
            -
            yfit_arr
        )

        sigma2_arr = (
            np.sum(
                residuals_arr ** 2
            )
            /
            (n_arr - 2)
            if n_arr > 2
            else 0
        )

        sigma_arr = (
            np.sqrt(
                sigma2_arr
            )
            if sigma2_arr > 0
            else 0
        )

        x_mean_arr = np.mean(
            x_arrhenius
        )

        sxx_arr = np.sum(
            (
                x_arrhenius
                -
                x_mean_arr
            ) ** 2
        )

        slope_error_arr = (
            sigma_arr
            /
            np.sqrt(sxx_arr)
            if sxx_arr > 0
            else 0
        )

        Ea_error = (
            kB
            *
            slope_error_arr
        )

        # ==================================================
        # RELATIVE ERRORS
        # ==================================================

        Ea_rel_error = (
            abs(
                Ea_error / Ea
            )
            *
            100
            if Ea != 0
            else 100
        )

        DeltaH_rel_error = (
            abs(
                DeltaH_error / DeltaH
            )
            *
            100
            if DeltaH != 0
            else 100
        )

        DeltaS_rel_error = (
            abs(
                DeltaS_error / DeltaS
            )
            *
            100
            if DeltaS != 0
            else np.inf
        )

        # ==================================================
        # VALIDITY
        # ==================================================

        validity = assess_validity(
            Ea_rel_error,
            DeltaH_rel_error,
            DeltaS_rel_error,
            A
        )

        # ==================================================
        # PRINT ARRHENIUS RESULTS
        # ==================================================

        print()

        print("=" * 75)

        print(
            "ARRHENIUS RESULTS"
        )

        print("=" * 75)

        print(
            f"Ea = "
            f"{Ea:.5f} +/- "
            f"{Ea_error:.5f} eV "
            f"({Ea_rel_error:.2f}%)"
        )

        print(
            f"A = "
            f"{A:.3e} s^-1 "
            f"(log10 = "
            f"{np.log10(A):.3f})"
        )

        print(
            f"R² = "
            f"{r_arrhenius**2:.6f}"
        )

        print(
            f"p-value = "
            f"{p_arrhenius:.4e}"
        )

        # ==================================================
        # PRINT EYRING RESULTS
        # ==================================================

        print()

        print("=" * 75)

        print(
            "EYRING RESULTS"
        )

        print("=" * 75)

        print(
            f"DeltaH‡ = "
            f"{DeltaH:.5f} +/- "
            f"{DeltaH_error:.5f} eV "
            f"({DeltaH_rel_error:.2f}%)"
        )

        print(
            f"DeltaS‡ = "
            f"{DeltaS:.6e} +/- "
            f"{DeltaS_error:.6e} eV/K "
            f"({DeltaS_rel_error:.2f}%)"
        )

        print(
            f"R² = "
            f"{r_eyring**2:.6f}"
        )

        print(
            f"p-value = "
            f"{p_eyring:.4e}"
        )

        # ==================================================
        # VALIDITY PRINT
        # ==================================================

        print()

        print("=" * 75)

        print(
            "VALIDITY ASSESSMENT"
        )

        print("=" * 75)

        print(
            f"Ea Quality      : "
            f"{validity['Ea_status']}"
        )

        print(
            f"DeltaH Quality  : "
            f"{validity['DeltaH_status']}"
        )

        print(
            f"DeltaS Quality  : "
            f"{validity['DeltaS_status']}"
        )

        print(
            f"A Status        : "
            f"{validity['A_status']}"
        )

        print(
            f"Overall         : "
            f"{validity['overall']}"
        )

        # ==================================================
        # PREDICTION LINES
        # ==================================================

        x_pred_eyring = np.linspace(
            min(x_eyring),
            max(x_eyring),
            300
        )

        y_pred_eyring = (
            slope_eyring
            *
            x_pred_eyring
            +
            intercept_eyring
        )

        x_pred_arrhenius = np.linspace(
            min(x_arrhenius),
            max(x_arrhenius),
            300
        )

        y_pred_arrhenius = (
            slope_arrhenius
            *
            x_pred_arrhenius
            +
            intercept_arrhenius
        )

        # ==================================================
        # ARRHENIUS PLOT
        # ==================================================

        fig1, ax1 = plt.subplots(
            figsize=(
                FIG_WIDTH,
                FIG_HEIGHT
            )
        )

        # --------------------------------------------------
        # DATA POINTS
        # --------------------------------------------------

        ax1.scatter(
            x_arrhenius,
            y_arrhenius,
            s=18,
            color=potential_color,
            edgecolors="none",
            alpha=1.0,
            zorder=3
        )

        # --------------------------------------------------
        # RED ARRHENIUS FIT
        # --------------------------------------------------

        ax1.plot(
            x_pred_arrhenius,
            y_pred_arrhenius,
            color="red",
            linewidth=1.5,
            alpha=1.0,
            zorder=4
        )

        # --------------------------------------------------
        # AXIS LABELS
        # --------------------------------------------------

        ax1.set_xlabel(
            r"$1/T$ (K$^{-1}$)",
            fontsize=12
        )

        ax1.set_ylabel(
            r"$\ln(\omega)$",
            fontsize=12
        )

        # ==================================================
        # UPPER-RIGHT IDENTIFICATION - FIXED
        # ==================================================

        # Cascade name at top

        ax1.text(
            0.98,
            0.97,
            f"{cascadeName}",
            transform=ax1.transAxes,
            fontsize=10,
            fontweight="bold",
            verticalalignment="top",
            horizontalalignment="right",
            alpha=1.0
        )

        # --------------------------------------------------
        # SMALL RED LINE - NOW BESIDE TEXT ON SAME LINE
        # --------------------------------------------------

        ax1.plot(
            [0.75, 0.80],           # ← RED LINE ON LEFT
            [0.90, 0.90],           # ← SAME y AS TEXT
            transform=ax1.transAxes,
            color="red",
            linewidth=1.5,
            solid_capstyle="butt",
            clip_on=False,
            zorder=5
        )

        # --------------------------------------------------
        # ARRHENIUS FIT TEXT - ON RIGHT SAME LINE
        # --------------------------------------------------

        ax1.text(
            0.98,                   # ← TEXT ON RIGHT
            0.90,                   # ← SAME y AS RED LINE
            "Arrhenius fit",
            transform=ax1.transAxes,
            fontsize=10,
            fontweight="bold",
            verticalalignment="center",
            horizontalalignment="right",
            alpha=1.0
        )

        # --------------------------------------------------
        # NO LEGEND
        # --------------------------------------------------

        legend = ax1.get_legend()

        if legend is not None:

            legend.remove()

        # --------------------------------------------------
        # GRID
        # --------------------------------------------------

        ax1.grid(
            True,
            linestyle="--",
            linewidth=0.5,
            color="lightgray",
            alpha=1.0
        )

        # --------------------------------------------------
        # TICKS
        # --------------------------------------------------

        ax1.tick_params(
            axis="both",
            which="major",
            direction="out",
            length=4,
            width=0.8,
            labelsize=10
        )

        # --------------------------------------------------
        # SPINES
        # --------------------------------------------------

        for spine in ax1.spines.values():

            spine.set_linewidth(
                0.8
            )

            spine.set_alpha(
                1.0
            )

        # --------------------------------------------------
        # LAYOUT
        # --------------------------------------------------

        plt.tight_layout()

        # --------------------------------------------------
        # SAVE ARRHENIUS
        # --------------------------------------------------

        arrhenius_base = os.path.join(
            PLOTS_DIR,
            f"{cascadeName}_Arrhenius_Plot"
        )

        save_publication_figure(
            fig1,
            arrhenius_base
        )

        plt.close(fig1)

        # ==================================================
        # EYRING PLOT
        # ==================================================

        fig2, ax2 = plt.subplots(
            figsize=(
                FIG_WIDTH,
                FIG_HEIGHT
            )
        )

        # --------------------------------------------------
        # DATA POINTS
        # --------------------------------------------------

        ax2.scatter(
            x_eyring,
            y_eyring,
            s=18,
            color=potential_color,
            edgecolors="none",
            alpha=1.0,
            zorder=3
        )

        # --------------------------------------------------
        # RED EYRING FIT
        # --------------------------------------------------

        ax2.plot(
            x_pred_eyring,
            y_pred_eyring,
            color="red",
            linewidth=1.5,
            alpha=1.0,
            zorder=4
        )

        # --------------------------------------------------
        # AXIS LABELS
        # --------------------------------------------------

        ax2.set_xlabel(
            r"$1/T$ (K$^{-1}$)",
            fontsize=12
        )

        ax2.set_ylabel(
            r"$\ln(\omega/T)$",
            fontsize=12
        )

        # ==================================================
        # UPPER-RIGHT IDENTIFICATION - FIXED
        # ==================================================

        # Cascade name at top

        ax2.text(
            0.98,
            0.97,
            f"{cascadeName}",
            transform=ax2.transAxes,
            fontsize=10,
            fontweight="bold",
            verticalalignment="top",
            horizontalalignment="right",
            alpha=1.0
        )

        # --------------------------------------------------
        # SMALL RED LINE - NOW BESIDE TEXT ON SAME LINE
        # --------------------------------------------------

        ax2.plot(
            [0.78, 0.85],           # ← RED LINE ON LEFT
            [0.90, 0.90],           # ← SAME y AS TEXT
            transform=ax2.transAxes,
            color="red",
            linewidth=1.5,
            alpha=1.0,
            solid_capstyle="butt",
            clip_on=False,
            zorder=5
        )

        # --------------------------------------------------
        # EYRING FIT TEXT - ON RIGHT SAME LINE
        # --------------------------------------------------

        ax2.text(
            0.98,                   # ← TEXT ON RIGHT
            0.90,                   # ← SAME y AS RED LINE
            "Eyring fit",
            transform=ax2.transAxes,
            fontsize=10,
            fontweight="bold",
            verticalalignment="center",
            horizontalalignment="right",
            alpha=1.0
        )

        # --------------------------------------------------
        # NO LEGEND
        # --------------------------------------------------

        legend = ax2.get_legend()

        if legend is not None:

            legend.remove()

        # --------------------------------------------------
        # GRID
        # --------------------------------------------------

        ax2.grid(
            True,
            linestyle="--",
            linewidth=0.5,
            color="lightgray",
            alpha=1.0
        )

        # --------------------------------------------------
        # TICKS
        # --------------------------------------------------

        ax2.tick_params(
            axis="both",
            which="major",
            direction="out",
            length=4,
            width=0.8,
            labelsize=10
        )

        # --------------------------------------------------
        # SPINES
        # --------------------------------------------------

        for spine in ax2.spines.values():

            spine.set_linewidth(
                0.8
            )

            spine.set_alpha(
                1.0
            )

        # --------------------------------------------------
        # LAYOUT
        # --------------------------------------------------

        plt.tight_layout()

        # --------------------------------------------------
        # SAVE EYRING
        # --------------------------------------------------

        eyring_base = os.path.join(
            PLOTS_DIR,
            f"{cascadeName}_Eyring_Plot"
        )

        save_publication_figure(
            fig2,
            eyring_base
        )

        plt.close(fig2)

        # ==================================================
        # STORE RESULTS
        # ==================================================

        all_results[cascadeName] = {

            "potential":
                potential,

            # ------------------------------------------------
            # Arrhenius
            # ------------------------------------------------

            "Ea_eV":
                float(Ea),

            "Ea_error_eV":
                float(Ea_error),

            "Ea_rel_error_percent":
                float(Ea_rel_error),

            "A":
                float(A),

            "log10_A":
                float(
                    np.log10(A)
                ),

            "r_squared_arrhenius":
                float(
                    r_arrhenius**2
                ),

            "p_value_arrhenius":
                float(
                    p_arrhenius
                ),

            # ------------------------------------------------
            # Eyring
            # ------------------------------------------------

            "DeltaH_eV":
                float(DeltaH),

            "DeltaH_error_eV":
                float(DeltaH_error),

            "DeltaH_rel_error_percent":
                float(DeltaH_rel_error),

            "DeltaS_eVK":
                float(DeltaS),

            "DeltaS_error_eVK":
                float(DeltaS_error),

            "DeltaS_rel_error_percent":
                float(DeltaS_rel_error),

            "r_squared_eyring":
                float(
                    r_eyring**2
                ),

            "p_value_eyring":
                float(
                    p_eyring
                ),

            # ------------------------------------------------
            # Temperature
            # ------------------------------------------------

            "avg_temperature_K":
                float(
                    np.mean(used_T)
                ),

            "n_temperatures":
                len(used_T),

            "n_trajectories":
                len(x_eyring),

            # ------------------------------------------------
            # Validity
            # ------------------------------------------------

            "valid":
                validity["valid"],

            "validity_overall":
                validity["overall"],

            "Ea_status":
                validity["Ea_status"],

            "DeltaH_status":
                validity["DeltaH_status"],

            "DeltaS_status":
                validity["DeltaS_status"],

            "A_status":
                validity["A_status"]

        }

        print(
            f"✓ Successfully processed "
            f"{cascadeName}"
        )

    # ======================================================
    # ERROR HANDLING
    # ======================================================

    except Exception as e:

        error_count += 1

        print(
            f"✗ Error processing "
            f"{cascadeName}: "
            f"{str(e)}"
        )

        continue


# ==========================================================
# SAVE SUMMARY RESULTS
# ==========================================================

print()

print("=" * 75)

print(
    "SUMMARY"
)

print("=" * 75)

print(
    f"Total cascades processed: "
    f"{processed_count}"
)

print(
    f"Successfully analyzed: "
    f"{len(all_results)}"
)

print(
    f"Skipped: "
    f"{skipped_count}"
)

print(
    f"Errors: "
    f"{error_count}"
)


# ==========================================================
# SAVE JSON
# ==========================================================

with open(
    "arrhenius_eyring_results.json",
    "w"
) as f:

    json.dump(
        all_results,
        f,
        indent=2,
        default=str
    )


# ==========================================================
# FINAL MESSAGE
# ==========================================================

print()

print(
    "Analysis complete!"
)

print()

print(
    "Plot identification:"
)


print()

print(
    "Point colors:"
)

print(
    "  DND-BN    = blue"
)

print(
    "  JW        = orange"
)

print(
    "  Marinica  = green"
)

print(
    "  Unknown   = black"
)

print()

print("=" * 75)

Using existing directory: Arrhenius_Eyring_Plots

STEP 2: ARRHENIUS/EYRING ANALYSIS ON EXTRACTED DATA
Input file size: 0.25 MB
Reading extracted_essential_data.json...
Total cascades found: 17


Processing (1/17): DND-BN_8_4_181_532
Potential: DND-BN (color: blue)
Total trajectories: 176

Temperature statistics:
---------------------------------------------------------------------------
  575 K : Total=  16 Valid=  16 Bad=  0 USE
  600 K : Total=  16 Valid=  16 Bad=  0 USE
  625 K : Total=  16 Valid=  16 Bad=  0 USE
  650 K : Total=  16 Valid=  16 Bad=  0 USE
  675 K : Total=  16 Valid=  16 Bad=  0 USE
  700 K : Total=  16 Valid=  16 Bad=  0 USE
  725 K : Total=  16 Valid=  16 Bad=  0 USE
  750 K : Total=  16 Valid=  16 Bad=  0 USE
  775 K : Total=  16 Valid=  16 Bad=  0 USE
  800 K : Total=  16 Valid=  16 Bad=  0 USE
  825 K : Total=  16 Valid=  16 Bad=  0 USE
Data points for fitting: 176

ARRHENIUS RESULTS
Ea = 0.47620 +/- 0.03735 eV (7.84%)
A = 7.177e+12 s^-1 (log10 = 12.856)
R² = 0

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os


# ================================================================
# STEP 3: DEFECT SIZE VS TRANSITION ENERGY PLOT
# USING MANUAL SIZE MAPPING
# ================================================================

print("=" * 80)
print("STEP 3: DEFECT SIZE VS TRANSITION ENERGY PLOT")
print("=" * 80)


# ================================================================
# FILE PATHS
# ================================================================

EXTRACTED_FILE = "extracted_essential_data.json"
RESULTS_FILE = "arrhenius_eyring_results.json"


# ================================================================
# MANUAL SIZE MAPPING
# ================================================================

SIZE_MAPPING = {

    # DND-BN
    "DND-BN_8_4_181_532": 8,
    "DND-BN_8_4_204_457": 8,
    "DND-BN_12_11_226_2": 12,
    "DND-BN_11_6_125_10": 11,
    "DND-BN_24_17_112_34": 24,
    "DND-BN_19_15_197_717": 19,
    "DND-BN_18_17_185_106": 21,

    # JW
    "JW_11_8_47_134": 11,
    "JW_6_6_44_118": 6,
    "JW_6_5_32_18": 6,
    "JW_6_5_29_18": 6,
    "JW_10_7_45_60": 10,
    "JW_7_6_27_20": 7,
    "JW_7_6_36_46": 7,

    # M-S
    "M-S_9_7_89_108": 9,
    "M-S_16_11_52_66": 16,
    "M-S_12_9_73_22": 12,
}


# ================================================================
# READ EXTRACTED DATA
# ================================================================

if not os.path.exists(EXTRACTED_FILE):

    print(f"ERROR: {EXTRACTED_FILE} not found!")
    print(f"Current directory: {os.getcwd()}")
    raise FileNotFoundError(EXTRACTED_FILE)


with open(EXTRACTED_FILE, "r") as f:

    extracted_data = json.load(f)


print(
    f"Loaded {len(extracted_data)} cascades "
    f"from {EXTRACTED_FILE}"
)


# ================================================================
# READ ARRHENIUS/EYRING RESULTS
# ================================================================

if not os.path.exists(RESULTS_FILE):

    print(f"ERROR: {RESULTS_FILE} not found!")
    print("Please run the Arrhenius/Eyring analysis first.")
    print(f"Current directory: {os.getcwd()}")
    raise FileNotFoundError(RESULTS_FILE)


with open(RESULTS_FILE, "r") as f:

    all_results = json.load(f)


print(
    f"Loaded {len(all_results)} results "
    f"from {RESULTS_FILE}"
)


# ================================================================
# HELPER FUNCTIONS
# ================================================================

def get_potential(cascade_name):

    name = cascade_name.lower()

    if (
        "dnd-bn" in name
        or "dndbn" in name
        or name.startswith("dnd")
    ):

        return "DND-BN"

    elif "jw" in name:

        return "JW"

    elif (
        "marinica" in name
        or "m-s" in name
        or "ms_" in name
        or name.startswith("ms")
    ):

        return "M-S"

    else:

        return "Unknown"


def get_size_from_mapping(cascade_name):

    if cascade_name in SIZE_MAPPING:

        return SIZE_MAPPING[cascade_name]

    parts = cascade_name.split("_")

    if len(parts) >= 2:

        try:

            return int(parts[1])

        except (ValueError, TypeError):

            pass

    return np.nan


# ================================================================
# CREATE DATA
# ================================================================

data_list = []


for cascade_name in extracted_data.keys():

    if cascade_name not in all_results:
        continue


    results = all_results[cascade_name]

    potential = get_potential(cascade_name)

    size = get_size_from_mapping(cascade_name)


    if np.isnan(size):
        continue


    # ------------------------------------------------------------
    # TRANSITION ENERGY
    # ------------------------------------------------------------

    Etr = results.get(
        "Ea_eV",
        np.nan
    )


    if np.isnan(Etr) or Etr <= 0:
        continue


    # ------------------------------------------------------------
    # STORE
    # ------------------------------------------------------------

    data_list.append({

        "Potential": potential,

        "Size": size,

        "Etr": Etr

    })


# ================================================================
# CHECK DATA
# ================================================================

if len(data_list) == 0:

    print("ERROR: No valid data found!")
    raise RuntimeError("No valid data found.")


# ================================================================
# DATAFRAME
# ================================================================

df = pd.DataFrame(data_list)


df = df.dropna(
    subset=[
        "Size",
        "Etr"
    ]
).copy()


df = df[
    df["Etr"] > 0
].copy()


df = df.sort_values(
    [
        "Potential",
        "Size"
    ]
).reset_index(
    drop=True
)


# ================================================================
# PRINT ONLY POTENTIAL, SIZE, ETR
# ================================================================

print()
print("=" * 60)
print("DATA USED FOR PLOTTING")
print("=" * 60)

print(
    df[
        [
            "Potential",
            "Size",
            "Etr"
        ]
    ].to_string(
        index=False
    )
)


# ================================================================
# PLOT SETTINGS
# ================================================================

potential_colors = {

    "DND-BN": "blue",

    "JW": "orange",

    "M-S": "green"

}


potential_markers = {

    "DND-BN": "o",

    "JW": "s",

    "M-S": "D"

}


potential_order = [

    "DND-BN",

    "JW",

    "M-S"

]


plt.rcParams.update({

    "font.family": "sans-serif",

    "font.sans-serif": [
        "Arial",
        "Helvetica",
        "DejaVu Sans"
    ],

    "font.size": 10,

    "axes.labelsize": 14,

    "xtick.labelsize": 11,

    "ytick.labelsize": 11,

    "axes.linewidth": 0.8,

    "figure.dpi": 300,

    "savefig.dpi": 600,

    "pdf.fonttype": 42,

    "ps.fonttype": 42

})


# ================================================================
# CREATE FIGURE
# ================================================================

fig, ax = plt.subplots(
    figsize=(8, 6)
)


# ================================================================
# PLOT EACH POTENTIAL
# ================================================================

for potential in potential_order:

    sub = df[
        df["Potential"] == potential
    ].sort_values(
        "Size"
    )


    if len(sub) == 0:
        continue


    # ------------------------------------------------------------
    # DATA POINTS
    # ------------------------------------------------------------

    ax.scatter(

        sub["Size"],

        sub["Etr"],

        color=potential_colors[
            potential
        ],

        marker=potential_markers[
            potential
        ],

        s=100,

        label=potential,

        edgecolor="black",

        linewidth=0.7,

        zorder=3

    )


    # ------------------------------------------------------------
    # CONNECTING RED LINE
    # ------------------------------------------------------------

    if len(sub) >= 2:

        ax.plot(

            sub["Size"],

            sub["Etr"],

            color="red",

            linewidth=1.5,

            linestyle="-",

            zorder=2

        )


# ================================================================
# AXIS LIMITS
# ================================================================

xmin = df["Size"].min()

xmax = df["Size"].max()


ax.set_xlim(

    xmin - 1,

    xmax + 1

)


ymin = df["Etr"].min()

ymax = df["Etr"].max()

yrange = ymax - ymin


if yrange > 0:

    ax.set_ylim(

        ymin - 0.10 * yrange,

        ymax + 0.10 * yrange

    )


# ================================================================
# AXIS LABELS
# ================================================================

ax.set_xlabel(

    "Defect Size",

    fontsize=14,

    fontweight="bold"

)


ax.set_ylabel(

    r"Transition Energy $E_{\mathrm{tr}}$ (eV)",

    fontsize=14,

    fontweight="bold"

)


# ================================================================
# LEGEND
# ================================================================

ax.legend(

    title="Potential",

    fontsize=11,

    title_fontsize=12,

    frameon=True

)


# ================================================================
# GRID
# ================================================================

ax.grid(

    True,

    which="major",

    alpha=0.25,

    linestyle="--"

)


# ================================================================
# TICKS
# ================================================================

ax.tick_params(

    axis="both",

    which="major",

    labelsize=11

)


# ================================================================
# LAYOUT
# ================================================================

plt.tight_layout()


# ================================================================
# SAVE PLOTS
# ================================================================

print()
print("=" * 80)
print("SAVING PLOTS")
print("=" * 80)


for fmt, ext in [

    ("eps", "eps"),

    ("pdf", "pdf"),

    ("png", "png")

]:

    filename = f"DefectSize_vs_Etr.{ext}"


    fig.savefig(

        filename,

        format=fmt,

        dpi=600 if fmt != "eps" else 300,

        bbox_inches="tight"

    )


    print(
        f"Saved: {filename}"
    )


# ================================================================
# FINAL MESSAGE
# ================================================================

print()
print("=" * 80)
print("PLOT GENERATION COMPLETE!")
print("=" * 80)


plt.show()

plt.close(fig)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


STEP 3: DEFECT SIZE VS TRANSITION ENERGY PLOT
Loaded 17 cascades from extracted_essential_data.json
Loaded 17 results from arrhenius_eyring_results.json

DATA USED FOR PLOTTING
Potential  Size      Etr
   DND-BN     8 0.476199
   DND-BN     8 0.648167
   DND-BN    11 0.687691
   DND-BN    12 0.638785
   DND-BN    19 1.157868
   DND-BN    21 1.616028
   DND-BN    24 1.471955
       JW     6 0.568056
       JW     6 0.181478
       JW     6 0.165188
       JW     7 0.338126
       JW     7 0.686857
       JW    10 0.305584
       JW    11 0.399355
      M-S     9 0.390540
      M-S    12 0.718251
      M-S    16 0.699290

SAVING PLOTS
Saved: DefectSize_vs_Etr.eps
Saved: DefectSize_vs_Etr.pdf
Saved: DefectSize_vs_Etr.png

PLOT GENERATION COMPLETE!


/tmp/ipykernel_9744/1182915229.py:589: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
import os
import json
import numpy as np
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
from scipy import stats
from collections import defaultdict
import pandas as pd


# ==========================================================
# CONSTANTS
# ==========================================================

kB = 8.617333262e-5       # eV/K
h = 4.135667696e-15       # eV*s


# ==========================================================
# INPUT
# ==========================================================

JSON_FILE = "extracted_essential_data.json"


# ==========================================================
# POTENTIAL COLORS
# ==========================================================

POTENTIAL_COLORS = {
    "DND-BN": "blue",
    "JW": "orange",
    "Marinica": "green"
}


# ==========================================================
# SIZE MAPPING
# ==========================================================

SIZE_MAPPING = {
    "DND-BN_8_4_181_532": 8,
    "DND-BN_8_4_204_457": 8,
    "DND-BN_12_11_226_2": 12,
    "DND-BN_11_6_125_10": 11,
    "DND-BN_24_17_112_34": 24,
    "DND-BN_19_15_197_717": 19,
    "DND-BN_18_17_185_106": 21,

    "JW_11_8_47_134": 11,
    "JW_6_6_44_118": 6,
    "JW_6_5_32_18": 6,
    "JW_6_5_29_18": 6,
    "JW_10_7_45_60": 10,
    "JW_7_6_27_20": 7,
    "JW_7_6_36_46": 7,

    "M-S_9_7_89_108": 9,
    "M-S_16_11_52_66": 16,
    "M-S_12_9_73_22": 12,
}


# ==========================================================
# IDENTIFY POTENTIAL
# ==========================================================

def identify_potential(cascadeName):

    name = cascadeName.lower()

    if "dnd-bn" in name or "dndbn" in name:
        return "DND-BN"

    elif "jw" in name:
        return "JW"

    elif "marinica" in name or "m-s" in name or "ms_" in name:
        return "Marinica"

    else:
        return "Unknown"


# ==========================================================
# GET DEFECT SIZE
# ==========================================================

def get_size(cascade_name):

    if cascade_name in SIZE_MAPPING:
        return SIZE_MAPPING[cascade_name]

    parts = cascade_name.split("_")

    if len(parts) >= 2:
        try:
            return int(parts[1])
        except:
            pass

    return np.nan


# ==========================================================
# MATPLOTLIB SETTINGS
# ==========================================================

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],

    "font.size": 10,

    "axes.labelsize": 12,
    "axes.titlesize": 12,

    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    "figure.dpi": 300,
    "savefig.dpi": 600,

    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,

    "axes.linewidth": 0.8,

    "axes.facecolor": "white",
    "figure.facecolor": "white",

    "lines.linewidth": 1.2,

    "ps.fonttype": 42,
    "pdf.fonttype": 42,

    "text.usetex": False
})


# ==========================================================
# READ JSON
# ==========================================================

with open(JSON_FILE, "r") as f:
    allData = json.load(f)


# ==========================================================
# PROCESS CASCADES
# ==========================================================

results_for_plot = []


for cascadeName, cascade in allData.items():

    potential = identify_potential(cascadeName)

    size = get_size(cascadeName)

    temps = cascade.get("temp", [])
    flipTimes = cascade.get("flipTime", [])

    if len(temps) == 0:
        continue


    # ------------------------------------------------------
    # GROUP BY TEMPERATURE
    # ------------------------------------------------------

    temp_data = defaultdict(list)

    for T, flipTime in zip(temps, flipTimes):

        try:
            temp_data[int(T)].append(flipTime)
        except:
            continue


    # ------------------------------------------------------
    # SELECT USABLE TEMPERATURES
    # ------------------------------------------------------

    used_T = []

    for T in sorted(temp_data.keys()):

        samples = temp_data[T]

        valid_count = sum(
            1
            for flip in samples
            if flip > 0 and np.isfinite(flip)
        )

        total = len(samples)

        bad = total - valid_count

        if bad <= 5:
            used_T.append(T)


    if len(used_T) < 3:
        continue


    # ------------------------------------------------------
    # BUILD FIT ARRAYS
    # ------------------------------------------------------

    x_eyring = []
    y_eyring = []

    x_arrhenius = []
    y_arrhenius = []


    for T in used_T:

        for flipTime in temp_data[T]:

            if flipTime > 0 and np.isfinite(flipTime):

                omega = 1.0 / flipTime

                # Eyring
                x_eyring.append(1.0 / T)
                y_eyring.append(np.log(omega / T))

                # Arrhenius
                x_arrhenius.append(1.0 / T)
                y_arrhenius.append(np.log(omega))


    if len(x_eyring) < 3:
        continue


    x_eyring = np.asarray(x_eyring, dtype=float)
    y_eyring = np.asarray(y_eyring, dtype=float)

    x_arrhenius = np.asarray(x_arrhenius, dtype=float)
    y_arrhenius = np.asarray(y_arrhenius, dtype=float)


    # ======================================================
    # EYRING FIT
    # ======================================================

    fit_eyring = stats.linregress(
        x_eyring,
        y_eyring
    )

    slope_eyring = fit_eyring.slope
    intercept_eyring = fit_eyring.intercept
    r_eyring = fit_eyring.rvalue


    # ======================================================
    # ARRHENIUS FIT
    # ======================================================

    fit_arrhenius = stats.linregress(
        x_arrhenius,
        y_arrhenius
    )

    slope_arrhenius = fit_arrhenius.slope
    intercept_arrhenius = fit_arrhenius.intercept
    r_arrhenius = fit_arrhenius.rvalue


    # ======================================================
    # EYRING PARAMETERS
    # ======================================================

    DeltaH = -kB * slope_eyring

    DeltaS = kB * (
        intercept_eyring - np.log(kB / h)
    )


    # ======================================================
    # ERROR IN DELTA S
    #
    # Standard error of Eyring intercept
    # DeltaS = kB * intercept + constant
    #
    # Therefore:
    # sigma_DeltaS = kB * sigma_intercept
    # ======================================================

    n = len(x_eyring)

    x_mean = np.mean(x_eyring)

    Sxx = np.sum(
        (x_eyring - x_mean) ** 2
    )

    residuals = (
        y_eyring
        - (
            slope_eyring * x_eyring
            + intercept_eyring
        )
    )

    if n > 2 and Sxx > 0:

        residual_variance = np.sum(
            residuals ** 2
        ) / (n - 2)

        intercept_error = np.sqrt(
            residual_variance *
            (
                1.0 / n
                + x_mean**2 / Sxx
            )
        )

        DeltaS_error = kB * intercept_error

    else:

        DeltaS_error = np.nan


    # ======================================================
    # ARRHENIUS PARAMETERS
    # ======================================================

    Ea = -slope_arrhenius * kB

    A = np.exp(intercept_arrhenius)

    log10_A = np.log10(A)


    # ======================================================
    # STORE
    # ======================================================

    if (
        not np.isnan(size)
        and np.isfinite(DeltaS)
        and np.isfinite(DeltaS_error)
        and np.isfinite(log10_A)
    ):

        results_for_plot.append({

            "Cascade": cascadeName,

            "Potential": potential,

            "Size": size,

            "ΔS‡ (eV/K)": DeltaS,

            "ΔS_error (eV/K)": DeltaS_error,

            "log10_A": log10_A,

            "Ea (eV)": Ea,

            "A (s^-1)": A,

            "R2_Arrhenius": r_arrhenius**2,

            "R2_Eyring": r_eyring**2

        })


# ==========================================================
# DATAFRAME
# ==========================================================

df = pd.DataFrame(results_for_plot)


# ==========================================================
# FILTER OUTSIDE 10^12 - 10^14 s^-1
# ==========================================================

df_outside = df[
    (df["log10_A"] < 12.0) |
    (df["log10_A"] > 14.0)
].copy()


df_outside = df_outside.sort_values(
    ["Potential", "Size"]
).reset_index(drop=True)


# ==========================================================
# PRINT ONLY OUTSIDE-RANGE CASCADES
# ==========================================================

print()
print("=" * 75)
print("CASCADES OUTSIDE THE TYPICAL PHONON FREQUENCY RANGE")
print("=" * 75)

print(
    f"Total cascades analysed : {len(df)}"
)

print(
    f"Outside range            : {len(df_outside)}"
)

print()


if len(df_outside) > 0:

    print(
        df_outside[
            [
                "Cascade",
                "Potential",
                "Size",
                "log10_A",
                "ΔS‡ (eV/K)",
                "ΔS_error (eV/K)"
            ]
        ].to_string(index=False)
    )

else:

    print("No cascades outside the range.")


# ==========================================================
# COMMON PLOT SETTINGS
# ==========================================================

potential_colors = {
    "DND-BN": "blue",
    "JW": "orange",
    "Marinica": "green"
}

potential_markers = {
    "DND-BN": "o",
    "JW": "s",
    "Marinica": "D"
}

potential_order = [
    "DND-BN",
    "JW",
    "Marinica"
]


# ==========================================================
# PLOT 1
# ENTROPY VS DEFECT SIZE
# ==========================================================

if len(df_outside) > 0:

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )


    for potential in potential_order:

        sub = df_outside[
            df_outside["Potential"] == potential
        ].sort_values("Size")


        if len(sub) == 0:
            continue


        ax.scatter(

            sub["Size"],

            sub["ΔS‡ (eV/K)"],

            color=potential_colors[potential],

            marker=potential_markers[potential],

            s=120,

            label=potential,

            edgecolor="black",

            linewidth=0.8,

            zorder=3
        )


        if len(sub) >= 2:

            ax.plot(

                sub["Size"],

                sub["ΔS‡ (eV/K)"],

                color=potential_colors[potential],

                linewidth=1.2,

                zorder=2
            )


    ax.set_xlabel(
        "Defect Size",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        r"Activation Entropy $\Delta S^\ddagger$ (eV/K)",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_title(
        r"Cascades outside the typical phonon range",
        fontsize=13
    )

    ax.legend(
        title="Potential",
        fontsize=11,
        title_fontsize=12
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.6
    )

    ax.axhline(
        y=0,
        color="black",
        linestyle="-",
        linewidth=0.8
    )

    ax.xaxis.set_major_locator(
        plt.MaxNLocator(integer=True)
    )

    plt.tight_layout()


    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range.eps",
        format="eps",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range.pdf",
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range.png",
        format="png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)


# ==========================================================
# PLOT 2
# ENTROPY VS DEFECT SIZE WITH ERROR BARS
# ==========================================================

if len(df_outside) > 0:

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )


    for potential in potential_order:

        sub = df_outside[
            df_outside["Potential"] == potential
        ].sort_values("Size")


        if len(sub) == 0:
            continue


        # --------------------------------------------------
        # ERROR BAR PLOT
        # --------------------------------------------------

        ax.errorbar(

            sub["Size"],

            sub["ΔS‡ (eV/K)"],

            yerr=sub["ΔS_error (eV/K)"],

            fmt=potential_markers[potential],

            color=potential_colors[potential],

            markersize=9,

            markeredgecolor="black",

            markeredgewidth=0.8,

            capsize=4,

            capthick=1.0,

            elinewidth=1.0,

            linestyle="none",

            label=potential,

            zorder=3
        )


        # --------------------------------------------------
        # CONNECT POINTS
        # --------------------------------------------------

        if len(sub) >= 2:

            ax.plot(

                sub["Size"],

                sub["ΔS‡ (eV/K)"],

                color=potential_colors[potential],

                linewidth=1.2,

                zorder=2
            )


    ax.set_xlabel(
        "Defect Size",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        r"Activation Entropy $\Delta S$ (eV/K)",
        fontsize=14,
        fontweight="bold"
    )

    
    ax.legend(
        title="Potential",
        fontsize=11,
        title_fontsize=12
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.6
    )

    ax.axhline(
        y=0,
        color="black",
        linestyle="-",
        linewidth=0.8
    )

    ax.xaxis.set_major_locator(
        plt.MaxNLocator(integer=True)
    )

    plt.tight_layout()


    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.eps",
        format="eps",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.pdf",
        format="pdf",
        dpi=600,
        bbox_inches="tight"
    )

    fig.savefig(
        "Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.png",
        format="png",
        dpi=600,
        bbox_inches="tight"
    )

    plt.close(fig)


# ==========================================================
# FINAL MESSAGE
# ==========================================================

print()
print("=" * 75)
print("PLOTS CREATED")
print("=" * 75)

print("1. Entropy_vs_DefectSize_OUTSIDE_typical_range.eps")
print("2. Entropy_vs_DefectSize_OUTSIDE_typical_range.pdf")
print("3. Entropy_vs_DefectSize_OUTSIDE_typical_range.png")

print()

print("4. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.eps")
print("5. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.pdf")
print("6. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.png")

print()
print("Analysis complete.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



CASCADES OUTSIDE THE TYPICAL PHONON FREQUENCY RANGE
Total cascades analysed : 17
Outside range            : 9

           Cascade Potential  Size   log10_A  ΔS‡ (eV/K)  ΔS_error (eV/K)
DND-BN_12_11_226_2    DND-BN    12 11.932576   -0.000378         0.000127
     JW_6_6_44_118        JW     6 15.234665    0.000358         0.000072
      JW_6_5_29_18        JW     6 11.838392   -0.000288         0.000071
      JW_7_6_27_20        JW     7 14.255521    0.000189         0.000072
      JW_7_6_36_46        JW     7 14.935154    0.000278         0.000067
     JW_10_7_45_60        JW    10 10.755980   -0.000581         0.000047
    M-S_9_7_89_108  Marinica     9 10.626452   -0.000648         0.000055
    M-S_12_9_73_22  Marinica    12 11.104638   -0.000561         0.000098
   M-S_16_11_52_66  Marinica    16 10.437569   -0.000723         0.000111


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.



PLOTS CREATED
1. Entropy_vs_DefectSize_OUTSIDE_typical_range.eps
2. Entropy_vs_DefectSize_OUTSIDE_typical_range.pdf
3. Entropy_vs_DefectSize_OUTSIDE_typical_range.png

4. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.eps
5. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.pdf
6. Entropy_vs_DefectSize_OUTSIDE_typical_range_ERRORBARS.png

Analysis complete.
